# Rede de emissões atribuídas à produção brasileira

## 1. Objetivo

Como se organiza a rede intersetorial de emissões atribuídas à produção brasileira, quais setores ocupam posições mais centrais e com maior diversidade de destinos, e em que medida os fluxos de emissões se concentram em determinados setores?

Usamos P, o VAB e o catálogo de setores exportados pela primeira etapa. Não recalculamos a MIP. Os agentes são setores agregados, não empresas. Distinguimos volume, posição estrutural, diversidade de destinos e dependência; nenhuma medida isolada demonstra a eficácia de uma intervenção econômica.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from redes import dados
from redes.redes import carregar_matriz_emissoes, carregar_setores, matriz_para_grafo
from redes.visualizacoes import figura_mapa_calor, figura_setor, figura_rede, exportar_pagina_redes
from redes.visualizacoes import plotar_emissoes_atividade
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 70)
from redes.redes import carregar_valor_adicionado


## 2. Inputs e definição da rede

`outputs/matriz_emissoes_producao_2015.csv` contém P, em Gg de CO₂; `outputs/setores_2015.csv` identifica os setores. O manifesto existente verifica seus hashes antes da leitura. São outputs regeneráveis: execute a exportação da etapa MIP se não existirem. Hashes não são atualizados automaticamente.

**i → j** significa emissões da atividade i atribuídas à demanda final pelo bem da atividade j, incluindo exportações. A matriz já contém efeitos diretos e indiretos; não representa transações diretas ou caminhos físicos do carbono.

Separamos P da rede intersetorial W, obtida zerando apenas a diagonal. A diagonal permanece registrada como atribuição intrassetorial. Mantemos todos os nós, inclusive isolados, e todas as arestas positivas, sem threshold nos cálculos. Carregamos também o VAB exportado, exclusivamente para a comparação econômica. Não carregamos C, produção bruta ou intensidades adicionais.

P = diag(γ) (I − A)⁻¹ diag(y), com A restrita aos insumos nacionais e y incluindo exportações. A soma da linha i recupera γᵢxᵢ, as emissões estimadas do próprio emissor; os efeitos indiretos distribuem esse total entre os destinos, sem criar emissões adicionais. A soma da coluna j agrega emissões brasileiras atribuídas à demanda final de j. A diagonal Pᵢᵢ identifica coincidência entre origem emissora e atividade da demanda final; pode incluir requisitos indiretos que passam por outros setores. Não equivale ao consumo intermediário do setor por ele mesmo.

`outputs/valor_adicionado_2015.csv` fornece o VAB, em R$ milhões de 2015, verificado pelo manifesto.


In [ ]:
P = carregar_matriz_emissoes("matriz_emissoes_producao_2015")
setores = carregar_setores("setores_mip_2015")
assert P.index.equals(setores.index) and P.shape == (67, 67)
vab = carregar_valor_adicionado("valor_adicionado_2015")
if set(vab.index) != set(P.index):
    raise ValueError("Os códigos do VAB devem coincidir com os setores de P.")
vab = vab.reindex(P.index)
entradas = pd.DataFrame([dados.entrada(i) for i in ["matriz_emissoes_producao_2015", "setores_mip_2015", "valor_adicionado_2015"]])
display(entradas[["id", "arquivo", "sha256"]])
print(f"P: {P.shape[0]} emissores × {P.shape[1]} destinos, em Gg de CO₂")
display(setores.to_frame())

## 3. Emissões próprias e participação no VAB

Para cada atividade, calculamos as emissões próprias pela soma da linha de P, incluindo a diagonal: `e_i = Σ_j P_ij = γ_i x_i`, em Gg de CO₂. A ordenação decrescente por e preserva a ordem original nos empates.

A participação nas emissões é `p_emissoes_i = e_i / Σ e`; a participação econômica é `p_vab_i = VAB_i / Σ VAB`. O índice `I_i = p_emissoes_i / p_vab_i` indica participação nas emissões maior que no VAB quando supera 1, e menor quando fica abaixo de 1. VAB não é literalmente PIB; o índice não demonstra causalidade ou eficiência técnica.

Os três painéis apresentam emissões próprias, índice e participação no VAB. Se todas as emissões forem zero, a participação nas emissões e o índice ficam indefinidos (NaN). O VAB deve ser finito e positivo em todas as atividades.


In [ ]:
# Emissões próprias: soma da linha de P, incluindo a diagonal.
# Origem: matriz P (67 × 67). Unidade: Gg de CO₂; recupera γ_i x_i.
emissoes_proprias = P.sum(axis=1)
contas_atividade = pd.DataFrame({
    "descricao": setores,
    "emissoes_proprias": emissoes_proprias,
})
# Empates preservam a ordem setorial da matriz de entrada.
contas_atividade = contas_atividade.sort_values("emissoes_proprias", ascending=False, kind="stable")


In [ ]:
# Participação econômica: fração do valor adicionado total, e não da produção bruta.
# O alinhamento usa códigos setoriais, independentemente da ordem do gráfico.
if not vab.index.is_unique or set(vab.index) != set(P.index):
    raise ValueError("Os códigos do VAB devem coincidir com os setores de P.")
if not np.isfinite(vab).all() or (vab <= 0).any():
    raise ValueError("O VAB deve ser finito e positivo em todas as atividades.")
contas_atividade["vab_r_milhao"] = vab
contas_atividade["participacao_vab"] = vab / vab.sum()

# Participação nas emissões próprias: fração do total brasileiro modelado em P.
total_emissoes_proprias = contas_atividade["emissoes_proprias"].sum()
contas_atividade["participacao_emissoes"] = (
    contas_atividade["emissoes_proprias"] / total_emissoes_proprias if total_emissoes_proprias > 0 else np.nan
)
# Índice adimensional: acima de 1, a participação nas emissões supera a do VAB.
contas_atividade["indice_emissoes_vab"] = (
    contas_atividade["participacao_emissoes"] / contas_atividade["participacao_vab"]
)
display(contas_atividade)


In [ ]:
figura_contas, eixos_contas = plotar_emissoes_atividade(contas_atividade)
# A página referencia este PNG: regenerar a figura atualiza a imagem exibida ao recarregar a página.
pasta_imagens = dados.RAIZ_PROJETO / "docs" / "imagens"
pasta_imagens.mkdir(parents=True, exist_ok=True)
figura_contas.savefig(pasta_imagens / "emissoes_vab.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.1. Distribuição das células de P

Os diagramas de caixas mostram como as emissões atribuídas se distribuem entre as células de P. Cada célula representa uma relação entre a atividade emissora e a atividade do produto destinado à demanda final, incluindo os requisitos diretos e indiretos de produção.

O painel esquerdo apresenta todos os elementos de P, incluindo zeros e diagonal, em Gg de CO₂. O painel direito apresenta o logaritmo decimal dos valores positivos, permitindo comparar diferentes ordens de grandeza. Os zeros permanecem no primeiro painel e no resumo estatístico; não entram no logaritmo. Uma coordenada −3 no segundo painel corresponde a 0,001 Gg, não a uma emissão negativa.

A caixa delimita os quartis de 25% e 75%, e a linha interna indica a mediana. Os bigodes alcançam os valores observados até 1,5 intervalo interquartil além da caixa. Os quartis são calculados por interpolação linear, separadamente em cada escala. Todos os pontos são exibidos e permitem consultar origem, destino e peso original.

O resumo estatístico acompanha os gráficos. Esta etapa descreve a distribuição, sem definir critérios de exclusão: todas as conexões positivas são mantidas nos cálculos da rede.


In [ ]:
# Uma observação por célula, preservando origem, destino, zeros e diagonal.
celulas_p = pd.DataFrame({
    "origem": np.repeat(P.index.to_numpy(), len(P.columns)),
    "destino": np.tile(P.columns.to_numpy(), len(P.index)),
    "peso_gg": P.to_numpy().ravel(),
})
celulas_p["descricao_origem"] = celulas_p["origem"].map(setores)
celulas_p["descricao_destino"] = celulas_p["destino"].map(setores)
celulas_p["diagonal"] = celulas_p["origem"] == celulas_p["destino"]
positivas = celulas_p["peso_gg"] > 0
celulas_p["log10_peso"] = np.nan
celulas_p.loc[positivas, "log10_peso"] = np.log10(celulas_p.loc[positivas, "peso_gg"])

# Limites calculados em cada escala; o logaritmo muda a distribuição de referência.
resumos_p = []
for escala, coluna in [("original_gg", "peso_gg"), ("log10_positivos", "log10_peso")]:
    valores = celulas_p[coluna].dropna()
    q1, mediana, q3 = valores.quantile([.25, .5, .75], interpolation="linear")
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    resumos_p.append({"escala": escala, "celulas": len(valores),
        "zeros": int((valores == 0).sum()) if escala == "original_gg" else 0,
        "minimo": valores.min(), "q1": q1, "mediana": mediana, "q3": q3,
        "maximo": valores.max(), "limite_inferior": inferior, "limite_superior": superior,
        "outliers_inferiores": int((valores < inferior).sum()),
        "outliers_superiores": int((valores > superior).sum())})
resumo_distribuicao_p = pd.DataFrame(resumos_p).set_index("escala")
display(resumo_distribuicao_p)


In [ ]:
from redes.visualizacoes import figura_distribuicao_p
figura_boxplot_p = figura_distribuicao_p(celulas_p)
display(figura_boxplot_p)


In [ ]:
# P é preservada. W representa somente atribuições entre atividades distintas.
diagonal = pd.Series(np.diag(P), index=P.index, name="diagonal")
valores = P.to_numpy(copy=True)
np.fill_diagonal(valores, 0)
W = pd.DataFrame(valores, index=P.index, columns=P.columns)
G = matriz_para_grafo(W)
nx.set_node_attributes(G, setores.to_dict(), "descricao")
np.testing.assert_allclose(W.to_numpy().sum() + diagonal.sum(), P.to_numpy().sum())
assert list(G) == P.index.tolist() and nx.number_of_selfloops(G) == 0

## 4. Estrutura geral e volumes

Graus contam relações; forças somam pesos. A força de saída identifica atribuições de um emissor a outros destinos; a força de entrada identifica emissões de outros setores atribuídas ao bem demandado. Ambas excluem a diagonal.

`emissoes_totais_i = Σ_j P_ij` inclui a diagonal. Não confundir um setor isolado em W com um setor sem emissões. A densidade é `m/[n(n−1)]`. Componentes fracas ignoram a direção; fortes exigem caminhos nos dois sentidos. Usamos essas medidas apenas para descrever a estrutura, pois efeitos indiretos tornam a rede muito densa.

In [ ]:
metricas = pd.DataFrame({"descricao": setores})
metricas["emissoes_totais"] = P.sum(axis=1)
metricas["diagonal"] = diagonal
metricas["forca_saida"] = pd.Series(dict(G.out_degree(weight="weight")))
metricas["forca_entrada"] = pd.Series(dict(G.in_degree(weight="weight")))
metricas["grau_saida"] = pd.Series(dict(G.out_degree()))
metricas["grau_entrada"] = pd.Series(dict(G.in_degree()))
metricas["participacao_total"] = metricas["emissoes_totais"] / metricas["emissoes_totais"].sum() if P.to_numpy().sum() else 0
metricas["participacao_intersetorial"] = metricas["forca_saida"] / metricas["forca_saida"].sum() if G.size(weight="weight") else 0
np.testing.assert_allclose(metricas["forca_saida"] + diagonal, P.sum(axis=1))
np.testing.assert_allclose(metricas["forca_entrada"] + diagonal, P.sum(axis=0))
resumo = pd.DataFrame([{
    "nos": len(G), "arestas": G.number_of_edges(), "densidade": nx.density(G),
    "peso_total_gg": P.to_numpy().sum(), "peso_intersetorial_gg": G.size(weight="weight"),
    "diagonal_gg": diagonal.sum(), "isolados": nx.number_of_isolates(G),
    "componentes_fracas": nx.number_weakly_connected_components(G),
    "componentes_fortes": nx.number_strongly_connected_components(G),
}], index=pd.Index(["producao"], name="rede"))
display(resumo)
display(metricas.sort_values("emissoes_totais", ascending=False).head(10))

## 5. Concentração entre emissores

Calculamos separadamente dois universos: emissões totais por emissor, incluindo diagonal, e atribuições intersetoriais por emissor, sem diagonal. Em ambos entram todos os setores, inclusive valores zero.

Se `s_i` é a participação do emissor, **HHI = Σ_i s_i²**: vale 1/n na distribuição igual e 1 na concentração em um emissor. O **Gini** compara diferenças entre todos os pares: `Σ_i Σ_j |e_i−e_j| / (2n Σ_i e_i)`. Sem correção amostral, seu máximo com n setores é `(n−1)/n`.

A curva de Lorenz ordena do menor para o maior emissor; a curva de participação acumulada ordena do maior para o menor. Top 5 e top 10 mostram a parcela emitida pelos maiores em cada universo. Por convenção computacional, se o peso total for zero, índices e participações valem zero; nesse caso não há distribuição positiva para interpretar.

In [ ]:
universos = {"total_com_diagonal": metricas["emissoes_totais"],
             "intersetorial_sem_diagonal": metricas["forca_saida"]}
concentracao = {}
curvas = []
for nome, serie in universos.items():
    valores = serie.to_numpy(dtype=float)
    n = len(valores)
    total = valores.sum()
    participacoes = valores / total if total else np.zeros(n)
    crescente = np.sort(participacoes)
    concentracao[nome] = {
        "peso_gg": total,
        "top5": np.sort(participacoes)[::-1][:5].sum(),
        "top10": np.sort(participacoes)[::-1][:10].sum(),
        "hhi": np.square(participacoes).sum(),
        "gini": np.abs(valores[:, None] - valores[None, :]).sum() / (2 * n * total) if total else 0,
    }
    curvas.append(pd.DataFrame({"universo": nome, "fracao_setores": np.arange(n + 1) / n,
        "lorenz": np.r_[0, np.cumsum(crescente)],
        "acumulado_maiores": np.r_[0, np.cumsum(crescente[::-1])]}))
concentracao = pd.DataFrame.from_dict(concentracao, orient="index").rename_axis("universo")
curvas = pd.concat(curvas, ignore_index=True)
display(concentracao)

## 6. Centralidade estrutural além do volume

PageRank ponderado segue ligações proporcionalmente ao peso, com amortecimento 0,85 e teletransporte uniforme. Nós sem saída distribuem probabilidade uniformemente. O escore soma 1.

No grafo original, destaca **destinos** que recebem atribuições de origens importantes. No grafo invertido, destaca **emissores** relacionados a destinos importantes. Não são quantidades de emissões. A diagonal permanece excluída.

Comparamos PageRank emissor com força de saída, e PageRank destino com força de entrada. Posto 1 é o maior valor; empates recebem posto médio. Uma diferença positiva `posto_forca − posto_pagerank` indica promoção pelo PageRank. Correlação dos postos mede concordância; se um ranking for constante, a correlação fica indefinida. Forte concordância limita o ganho de informação em relação ao volume.

[Definição e parâmetros do NetworkX](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html). Não calculamos caminhos mínimos ou um índice agregado de criticidade.

As denominações PageRank emissor e destino designam, respectivamente, o grafo invertido e o original. A ponderação é normalizada a cada passo, e o teletransporte atribui escore positivo inclusive a isolados. Caminhos sobre W concatenam atribuições já acumuladas pela inversa de Leontief: o PageRank é uma descrição estrutural exploratória, não uma simulação de circulação de carbono nem de choques na produção. A correlação calculada é a correlação de Spearman (Pearson dos postos médios), sem teste de significância.


In [ ]:
metricas["pagerank_destino"] = pd.Series(nx.pagerank(G, alpha=.85, weight="weight", max_iter=1000, tol=1e-12))
metricas["pagerank_emissor"] = pd.Series(nx.pagerank(G.reverse(copy=False), alpha=.85, weight="weight", max_iter=1000, tol=1e-12))
np.testing.assert_allclose(metricas[["pagerank_destino", "pagerank_emissor"]].sum(), [1, 1])
rankings = metricas[["emissoes_totais", "forca_saida", "forca_entrada", "pagerank_emissor", "pagerank_destino"]].rank(ascending=False, method="average")
rankings["promocao_emissor"] = rankings["forca_saida"] - rankings["pagerank_emissor"]
rankings["promocao_destino"] = rankings["forca_entrada"] - rankings["pagerank_destino"]
rankings.insert(0, "descricao", setores)
for papel, forca in [("emissor", "forca_saida"), ("destino", "forca_entrada")]:
    a, b = rankings[forca], rankings["pagerank_" + papel]
    resumo["correlacao_rank_" + papel] = a.corr(b) if a.nunique() > 1 and b.nunique() > 1 else np.nan
display(rankings.sort_values("pagerank_emissor").head(10))
display(rankings.reindex(rankings["promocao_emissor"].abs().sort_values(ascending=False).index).head(10))
display(resumo[["correlacao_rank_emissor", "correlacao_rank_destino"]])

## 7. Diversidade de destinos e participação por origem

### 7.1. Distribuição das saídas

`q_ij = W_ij / Σ_j W_ij` descreve como cada emissor distribui suas atribuições. O **número efetivo de destinos = 1 / Σ_j q_ij²** vale 1 para concentração em um destino e k para k destinos igualmente relevantes. Um emissor sem saídas recebe zero. Essa diversidade de destinos distingue setores com o mesmo grau e volume, mas distribuições diferentes; não mede alcançabilidade por caminhos.

### 7.2. Participação dos emissores por destino

`d_ij = W_ij / Σ_i W_ij` mede a participação de i nas atribuições intersetoriais recebidas por j. Cada coluna não nula soma 1; colunas sem entradas ficam zeradas. Isso não é a parcela de insumos físicos comprada de i. Um destino pode apresentar alta dependência relativa com pequeno volume absoluto, por isso mostramos participação e peso juntos.

Para cada destino, listamos os três principais emissores; para cada emissor, os três destinos com maior dependência relativa. A seleção é apenas uma síntese das tabelas: a matriz completa de dependência é exportada. Empates nos recortes são ordenados pelo código setorial.

O número efetivo de destinos é o inverso de Simpson, ou número de Hill de ordem 2 ([Hill, 1973](https://doi.org/10.2307/1934352)); equivale ao inverso da disparidade dos pesos de saída ([Antoniou e Tsompa, 2008](https://doi.org/10.1155/2008/375452)). O nome “destinos” é uma adaptação à matriz de atribuições. Para saídas positivas, 1 ≤ Nᵢ ≤ grau de saída; zero representa ausência de distribuição e é uma convenção de armazenamento. Como Wᵢⱼ = γᵢLᵢⱼyⱼ para i ≠ j, γᵢ cancela na normalização por linha. Assim, a diversidade depende da estrutura de requisitos e da composição da demanda final, não da intensidade de emissão do próprio setor.

A razão dᵢⱼ é uma participação nas atribuições recebidas, não dependência de fornecimento ou vulnerabilidade a interrupções. Os nomes `dependencia_producao`, `maior_dependencia_destino` e `destinos_dependentes_por_emissor` são identificadores dos CSVs para essa participação. Não há estimativa de substituição de insumos, exposição a choques ou causalidade.


In [ ]:
q = W.div(metricas["forca_saida"].replace(0, np.nan), axis=0).fillna(0)
hhi_destinos = q.pow(2).sum(axis=1)
metricas["destinos_efetivos"] = 1 / hhi_destinos.replace(0, np.nan)
metricas["destinos_efetivos"] = metricas["destinos_efetivos"].fillna(0)
d = W.div(metricas["forca_entrada"].replace(0, np.nan), axis=1).fillna(0)
np.testing.assert_allclose(d.sum(axis=0), (metricas["forca_entrada"] > 0).astype(float))
np.testing.assert_allclose(q.sum(axis=1), (metricas["forca_saida"] > 0).astype(float))
metricas["maior_dependencia_destino"] = d.max(axis=1)
rankings["destinos_efetivos"] = metricas["destinos_efetivos"].rank(ascending=False, method="average")
principais_emissores = []
destinos_dependentes = []
for destino in d.columns:
    for emissor, participacao in d[destino].sort_index().sort_values(ascending=False, kind="stable").head(3).items():
        if participacao > 0:
            principais_emissores.append({"destino": destino, "descricao_destino": setores[destino],
                "emissor": emissor, "descricao_emissor": setores[emissor], "participacao": participacao, "peso_gg": W.loc[emissor, destino]})
for emissor in d.index:
    for destino, participacao in d.loc[emissor].sort_index().sort_values(ascending=False, kind="stable").head(3).items():
        if participacao > 0:
            destinos_dependentes.append({"emissor": emissor, "descricao_emissor": setores[emissor],
                "destino": destino, "descricao_destino": setores[destino], "participacao": participacao, "peso_gg": W.loc[emissor, destino]})
principais_emissores = pd.DataFrame(principais_emissores)
destinos_dependentes = pd.DataFrame(destinos_dependentes)
display(metricas[["descricao", "emissoes_totais", "forca_saida", "destinos_efetivos"]].sort_values("destinos_efetivos", ascending=False).head(10))
display(principais_emissores)
display(destinos_dependentes)

## 8. Visualizações e exploração

Lorenz e participação acumulada distinguem os dois universos de concentração. A dispersão relaciona volume total e diversidade de destinos, mantendo os setores sem saídas. Os mapas de W e d seguem a mesma ordem setorial: magnitude em `log10(1 + Gg)` e participação por origem de 0 a 100%, com diagonal mascarada e valores originais no hover.

Na exploração por setor mostramos até dez entradas e saídas, calculadas sobre toda W, com cobertura e diagonal. O desenho não pressupõe equilíbrio das entradas e saídas nem conservação de fluxos pelo setor central.

A visão geral apresenta as 20 maiores atividades por emissões próprias e o grupo Outras, em duas figuras separadas: circular e por forças. A agregação é detalhada a seguir.
A área dos nós é proporcional às emissões próprias agregadas: área = `2500 emissões / máximo`, em pontos quadrados, sem acréscimo fixo. A escala de tamanho é comum às duas figuras. Nós com zero emissões têm área zero.

As redes são desenhadas pelo NetworkX com Matplotlib e exportadas como PNGs. Cada conexão e sua ponta formam uma única aresta dirigida; os arquivos são utilizados também pela página de exploração.


### 8.1. Rede visual: 20 maiores emissores e Outras

Selecionamos as 20 maiores atividades pela soma da linha de P, incluindo a diagonal (emissões próprias, e não produção monetária ou soma da linha preta). Empates seguem o código. As demais atividades formam **Outras**, sempre ao final da ordem circular.

Somamos blocos de P por origem e destino. Isso preserva o total de emissões. A diagonal agregada de Outras inclui tanto as diagonais originais quanto as atribuições entre atividades do grupo. Essas parcelas são registradas como internas ao nó, mas não desenhadas como arestas. A tabela de cobertura informa a parcela intersetorial original retida nas ligações visíveis.

Tamanho dos nós: emissões próprias agregadas. Cor: diversidade de destinos recalculada nas saídas entre grupos. A diversidade muda com a agregação e não é média das diversidades originais. As demais análises e rankings continuam usando as 67 atividades. As duas figuras separadas usam os mesmos nós e pesos; o layout por forças ignora pesos e direção apenas para posicionar os nós, com semente 42.

A área dos nós é proporcional às emissões próprias agregadas: área = `2500 emissões / máximo`, em pontos quadrados, sem acréscimo fixo. A escala de tamanho é comum às duas figuras. Nós com zero emissões têm área zero.

As redes são desenhadas pelo NetworkX com Matplotlib e exportadas como PNGs. Cada conexão e sua ponta formam uma única aresta dirigida; os arquivos são utilizados também pela página de exploração.


In [ ]:
redes_visuais = {}
for limite in [20]:
    # Seleção pelo total emitido na origem, incluindo a diagonal; desempate por código.
    maiores_emissores = P.sum(axis=1).sort_index().sort_values(ascending=False, kind="stable").head(limite).index
    agrupamento = pd.Series([codigo if codigo in maiores_emissores else "Outras" for codigo in P.index], index=P.index)
    ordem_visual = maiores_emissores.tolist()
    if "Outras" in agrupamento.values:
        ordem_visual.append("Outras")
    
    # Soma dos blocos: linhas e colunas seguem a mesma classificação de grupos.
    P_visual = P.groupby(agrupamento, sort=False).sum()
    P_visual = P_visual.T.groupby(agrupamento, sort=False).sum().T
    P_visual = P_visual.reindex(index=ordem_visual, columns=ordem_visual)
    np.testing.assert_allclose(P_visual.to_numpy().sum(), P.to_numpy().sum())
    diagonal_visual = pd.Series(np.diag(P_visual), index=P_visual.index)
    valores_visuais = P_visual.to_numpy(copy=True)
    np.fill_diagonal(valores_visuais, 0)
    W_visual = pd.DataFrame(valores_visuais, index=ordem_visual, columns=ordem_visual)
    G_visual = matriz_para_grafo(W_visual)
    
    # Indicadores exclusivos do desenho agregado; não substituem as métricas originais.
    metricas_visuais = pd.DataFrame(index=P_visual.index)
    metricas_visuais["descricao"] = setores.reindex(P_visual.index)
    if "Outras" in metricas_visuais.index:
        metricas_visuais.loc["Outras", "descricao"] = f"Outras ({(agrupamento == 'Outras').sum()} atividades)"
    metricas_visuais["emissoes_totais"] = P_visual.sum(axis=1)
    metricas_visuais["diagonal"] = diagonal_visual
    metricas_visuais["forca_saida"] = W_visual.sum(axis=1)
    metricas_visuais["forca_entrada"] = W_visual.sum(axis=0)
    q_visual = W_visual.div(metricas_visuais["forca_saida"].replace(0, np.nan), axis=0).fillna(0)
    metricas_visuais["destinos_efetivos"] = (1 / q_visual.pow(2).sum(axis=1).replace(0, np.nan)).fillna(0)
    # Relações antes intersetoriais que passam a ser internas ao grupo Outras.
    peso_interno_outras = diagonal_visual.sum() - np.diag(P).sum()
    np.testing.assert_allclose(W_visual.to_numpy().sum() + peso_interno_outras, P.to_numpy().sum() - np.trace(P))
    redes_visuais[limite] = {"P": P_visual, "W": W_visual, "grafo": G_visual,
        "metricas": metricas_visuais, "peso_interno_outras": peso_interno_outras}


In [ ]:
descricoes_figuras = {
    "Curva de Lorenz": "Os setores são ordenados do menor para o maior volume de emissões. A curva relaciona a fração acumulada dos setores à fração acumulada das emissões; o afastamento abaixo da linha de igualdade representa concentração; curvas que se cruzam não permitem ordenação inequívoca apenas pela inspeção visual. Comparam-se as emissões totais, incluindo as atribuições ao próprio setor, e as emissões atribuídas a outros setores. Todos os setores entram no cálculo, inclusive aqueles com valor zero.",
    "Participação acumulada dos maiores": "Os setores são ordenados do maior para o menor volume de emissões em cada distribuição. A curva mostra a parcela das emissões concentrada em determinada fração dos maiores emissores. Uma elevação mais rápida indica maior concentração. As duas séries distinguem as emissões totais das atribuições exclusivamente intersetoriais.",
    "Volume e diversidade dos destinos": "Cada ponto representa um setor. O eixo horizontal apresenta suas emissões totais, incluindo as atribuições ao próprio setor. O eixo vertical mostra o número efetivo de destinos, calculado pelo inverso de Simpson (número de Hill de ordem 2), 1/Σq², usando as participações nas saídas intersetoriais: k destinos com participações iguais correspondem a um alcance efetivo de k. Setores sem saídas recebem zero por convenção; a distribuição não está definida nesse caso. A medida não expressa intensidade de emissão nem alcance por caminhos. A cor indica a maior participação desse emissor nas emissões intersetoriais atribuídas a um único destino.",
    "Magnitude intersetorial": "As linhas representam os setores emissores e as colunas, os setores cujos produtos atendem à demanda final. Cada célula contém as emissões da origem atribuídas ao destino, em Gg de CO₂. As cores utilizam log₁₀(1 + valor) para permitir a leitura de magnitudes distintas; a consulta interativa apresenta os valores originais. A diagonal foi omitida por corresponder às atribuições intrassetoriais. As relações incluem efeitos diretos e indiretos da produção.",
    "Participação dos emissores por destino": "Cada célula mostra a participação do emissor nas emissões intersetoriais atribuídas ao destino. O cálculo divide o peso da relação pelo total recebido de outros setores; colunas sem entradas são zeradas por convenção; cada coluna com entradas soma 100%. A ordem dos setores é a mesma do mapa de magnitude, e a diagonal está omitida. Uma participação elevada indica concentração relativa em uma origem, mesmo quando o volume absoluto é pequeno; não representa a participação desse emissor nas compras de insumos do destino.",
    "Explorar um setor": "Para o setor selecionado, são apresentadas até dez origens de emissões atribuídas à demanda final por seu produto, à esquerda, e até dez destinos das emissões geradas por ele, à direita. As relações são selecionadas pelo maior peso, e a largura das faixas representa as emissões em Gg de CO₂. Os percentuais informam a cobertura dos recortes em relação às entradas e saídas completas. A diagonal, informada separadamente, associa emissões do setor à demanda final de sua própria atividade, inclusive via requisitos indiretos. Um setor pode aparecer dos dois lados. Entradas e saídas representam atribuições distintas e não pressupõem conservação de fluxo pelo setor central."
}

figuras = {}
figuras["Curva de Lorenz"] = px.line(curvas, x="fracao_setores", y="lorenz", color="universo", template="plotly_white",
    labels={"fracao_setores": "Fração dos setores (menor → maior)", "lorenz": "Fração das emissões"})
figuras["Curva de Lorenz"].add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color="gray", dash="dot"))
figuras["Participação acumulada dos maiores"] = px.line(curvas, x="fracao_setores", y="acumulado_maiores", color="universo", template="plotly_white",
    labels={"fracao_setores": "Fração dos setores (maior → menor)", "acumulado_maiores": "Fração das emissões"})
for figura in figuras.values():
    figura.update_layout(legend=dict(orientation="h", y=-.35), margin=dict(b=125), height=520)
figuras["Volume e diversidade dos destinos"] = px.scatter(metricas.reset_index(), x="emissoes_totais", y="destinos_efetivos",
    color="maior_dependencia_destino", hover_name="descricao", hover_data=["atividade", "forca_saida", "pagerank_emissor", "diagonal"],
    labels={"emissoes_totais": "Emissões totais (Gg, incluindo diagonal)", "destinos_efetivos": "Número efetivo de destinos",
            "maior_dependencia_destino": "Maior participação<br>em um destino"}, template="plotly_white")
figuras["Volume e diversidade dos destinos"].update_layout(coloraxis=dict(cmin=0, cmax=1, colorbar=dict(tickformat=".0%")))
max_peso = float(W.to_numpy().max())
figuras["Magnitude intersetorial"] = figura_mapa_calor(W, setores, "Emissões atribuídas (Gg)", max_peso)
figuras["Participação dos emissores por destino"] = figura_mapa_calor(d, setores, "Participação no destino", 1, dependencia=True)

estados = []
for setor in setores.index:
    entradas_setor = sorted(G.in_edges(setor, data=True), key=lambda e: (-e[2]["weight"], e[0]))[:10]
    saidas_setor = sorted(G.out_edges(setor, data=True), key=lambda e: (-e[2]["weight"], e[1]))[:10]
    total_entrada, total_saida = metricas.loc[setor, ["forca_entrada", "forca_saida"]]
    estados.append({"setor": setor, "descricao": setores[setor],
        "entradas": [(i, setores[i], a["weight"]) for i, _, a in entradas_setor],
        "saidas": [(j, setores[j], a["weight"]) for _, j, a in saidas_setor],
        "cobertura_entrada": 100 * sum(a["weight"] for _, _, a in entradas_setor) / total_entrada if total_entrada else 0,
        "cobertura_saida": 100 * sum(a["weight"] for _, _, a in saidas_setor) / total_saida if total_saida else 0,
        "diagonal": diagonal[setor]})
figuras["Explorar um setor"] = figura_setor(estados, "Entradas → setor selecionado → saídas")
# Escalas comuns de tamanho e espessura nas duas disposições.
max_emissoes_visual = max(rede["metricas"]["emissoes_totais"].max() for rede in redes_visuais.values())
max_peso_visual = max(rede["W"].to_numpy().max() for rede in redes_visuais.values())
figuras_redes = {}
coberturas = []
for limite, rede in redes_visuais.items():
    grafo_visual = rede["grafo"]
    arestas_visuais = list(grafo_visual.edges(data=True))
    coberturas.append({"recorte": f"{limite} maiores + Outras",
        "nos_exibidos": len(grafo_visual), "arestas_exibidas": len(arestas_visuais),
        "arestas_calculo": G.number_of_edges(), "peso_interno_outras_gg": rede["peso_interno_outras"],
        "peso_exibido_pct": 100 * grafo_visual.size(weight="weight") / G.size(weight="weight") if G.size(weight="weight") else 0})
    for organizacao, nome in [("circular", "Circular"), ("forcas", "Por forças")]:
        titulo = f"{nome} · {limite} maiores emissores e Outras"
        figura_rede_estatica, eixo_rede = figura_rede(grafo_visual, rede["metricas"], titulo,
            max_peso_visual, max_emissoes_visual, organizacao=organizacao)
        caminho_imagem = Path("imagens") / f"rede_{organizacao}.png"
        (dados.RAIZ_PROJETO / "docs" / "imagens").mkdir(parents=True, exist_ok=True)
        figura_rede_estatica.savefig(dados.RAIZ_PROJETO / "docs" / caminho_imagem, dpi=180, bbox_inches="tight")
        figuras_redes[titulo] = caminho_imagem
        display(figura_rede_estatica)
        plt.close(figura_rede_estatica)
        descricoes_figuras[titulo] = (
            f"As {limite} maiores atividades pela soma das linhas de P são mantidas; as demais formam Outras. "
            "Pesos são somados entre grupos. A área dos nós é proporcional às emissões próprias agregadas, "
            "com a mesma escala nas duas figuras (área máxima de 2.500 pontos quadrados). A cor indica a diversidade "
            "de destinos recalculada entre grupos, na escala indicada na própria figura. As atribuições internas "
            "aos nós ficam na diagonal separada; isso inclui as relações entre atividades do grupo Outras. "
            + ("A disposição circular segue o ranking, com Outras ao final." if organizacao == "circular" else
               "O layout por forças usa todas as ligações entre grupos, sem pesos de atração, ignorando a direção para posicionar os nós e com semente 42. Distâncias não medem similaridade econômica."))
cobertura = pd.DataFrame(coberturas).set_index("recorte")
descricoes_figuras["Distribuição das células de P"] = (
    f"A distribuição reúne os {len(celulas_p)} elementos de P, incluindo {int((celulas_p['peso_gg'] == 0).sum())} zeros e toda a diagonal. "
    "Cada célula representa emissões de uma atividade atribuídas à demanda final pelo produto de outra ou da própria atividade. "
    "À esquerda, os pesos são apresentados em Gg de CO₂; à direita, o logaritmo decimal dos valores positivos permite comparar ordens de grandeza. "
    "Os zeros não entram no logaritmo. Uma coordenada −3 nesse painel corresponde a 0,001 Gg, não a uma emissão negativa. "
    "A caixa abrange os quartis de 25% a 75%, com a mediana ao centro; os bigodes alcançam os valores observados até 1,5 intervalo interquartil além da caixa. "
    "O cálculo é realizado separadamente em cada escala. Passe o mouse sobre os pontos para consultar origem, destino e peso. "
    "O resumo estatístico aparece imediatamente abaixo. A análise é descritiva: todas as conexões positivas são mantidas nos cálculos da rede.")
figuras = {"Distribuição das células de P": figura_boxplot_p, **figuras_redes, **figuras}
display(cobertura)
for nome, figura in figuras.items():
    if nome != "Distribuição das células de P" and not isinstance(figura, Path): display(figura)

notas_metodologicas = {
    "Construção e unidade": "P = diag(γ) (I − A)⁻¹ diag(y). A contém os coeficientes de insumos nacionais; y é a demanda final por produtos nacionais, incluindo exportações; γ é a intensidade estimada de CO₂. A rede W mantém os elementos fora da diagonal. A soma de uma linha de P é a emissão estimada do próprio setor (γᵢxᵢ); a coluna reúne emissões brasileiras atribuídas à demanda final do destino. Gg equivale a mil toneladas. As setas representam atribuições contábeis, não transações diretas ou transporte de carbono.",
    "Diversidade dos destinos": "Nᵢ = 1/Σⱼqᵢⱼ², com qᵢⱼ = Wᵢⱼ/ΣⱼWᵢⱼ. Trata-se do inverso de Simpson (número de Hill de ordem 2), aplicado às saídas. “Número efetivo de destinos” é o nome adotado nesta aplicação. Para saídas positivas, varia de 1 ao grau de saída; zero indica ausência de distribuição, por convenção. A intensidade γᵢ cancela na normalização: a medida descreve a distribuição dos pesos, não intensidade de carbono ou alcance por caminhos.",
    "Participação por origem": "dᵢⱼ = Wᵢⱼ/ΣᵢWᵢⱼ. O denominador exclui a diagonal. A participação mede concentração relativa das origens nas atribuições do destino, não dependência econômica ou risco de interrupção. Nos arquivos, os identificadores com “dependencia” correspondem a essa razão. As participações são frações de 0 a 1 nos CSVs; o mapa apresenta percentuais.",
    "Centralidade e rankings": "Força de saída é a soma da linha de W; força de entrada, a soma da coluna. Grau conta relações positivas. PageRank usa pesos, amortecimento 0,85 e distribuição uniforme no teletransporte e nos nós sem saída. “Destino” usa o grafo original e “emissor”, o invertido; ambos somam 1 e incluem escores positivos para isolados. Como W já inclui efeitos indiretos, percursos do PageRank não representam cadeias físicas adicionais. É uma centralidade exploratória, não uma estimativa de emissões evitadas. Posto 1 indica o maior valor; empates recebem posto médio. A correlação dos rankings é de Spearman, sem teste de significância.",
    "Concentração": "O HHI soma os quadrados das participações e usa escala de 0 a 1, não de 0 a 10.000. Para n setores e total positivo, varia de 1/n a 1. O Gini compara diferenças entre pares, sem correção amostral, e tem máximo (n−1)/n. Os dois universos distinguem a inclusão e a exclusão da diagonal. Se não houver emissões, os zeros armazenados são convenções; não indicam uma distribuição igualitária observada.",
    "Hipóteses e limites": "A estrutura produtiva é a MIP de 2015, com 67 atividades. As intensidades resultam da interpolação linear entre os coeficientes arredondados de 2011 e 2018 de Sanguinet e Azzoni. A etapa MIP adota fator monetário 1 entre os coeficientes em base de preços de 2018 e a produção a preços de 2015, sem deflação efetiva; os níveis absolutos são aproximações. A rede cobre as emissões brasileiras modeladas, não as emissões externas incorporadas nas importações nem um inventário territorial completo. As figuras são descritivas: não testam formalmente as hipóteses de concentração nas ligações ou exposição de setores pouco emissores. As curvas de Lorenz medem concentração entre emissores, não entre ligações. Centralidade e participação elevada não demonstram o efeito de uma intervenção."
}

# Entrada completa: 67 × 67 células, em Gg de CO₂, incluindo a diagonal.
figuras["Matriz P completa"] = figura_mapa_calor(P, setores, "Matriz P completa", float(P.to_numpy().max()), incluir_diagonal=True)
descricoes_figuras.update({'Matriz P completa': 'Linhas: atividades emissoras; colunas: destinos da demanda final. A cor representa log₁₀(1 + emissões em Gg de CO₂); consulte os valores originais ao passar o mouse.', 'Participação acumulada dos maiores': 'Setores da maior para a menor emissão. Quanto mais rápida a subida, maior a concentração.', 'Volume e diversidade dos destinos': 'Destinos efetivos: k destinos com pesos iguais equivalem a k. Quanto mais uniforme a distribuição, maior a diversidade. A cor indica a maior participação do emissor nas atribuições recebidas por um destino.', 'Curva de Lorenz': 'Setores da menor para a maior emissão. Maior afastamento da linha de igualdade indica maior concentração; curvas cruzadas não permitem ordenação direta.', 'Distribuição das células de P': 'Todas as células, incluindo a diagonal. No logaritmo, apenas valores positivos: posições negativas significam menos de 1 Gg. Nenhuma conexão é excluída.', 'Magnitude intersetorial': 'P sem diagonal. Cor em log₁₀(1 + valor); valores originais em Gg ao passar o mouse.', 'Participação dos emissores por destino': 'Sem diagonal. Cada coluna com entradas soma 100%: maior percentual indica maior concentração naquela origem.', 'Explorar um setor': 'Até dez origens e dez destinos, com cobertura percentual e diagonal separada.'})

# Rótulos públicos descrevem os universos sem expor identificadores dos CSVs.
for nome in ["Curva de Lorenz", "Participação acumulada dos maiores"]:
    figuras[nome].update_layout(legend_title_text="")
    for serie in figuras[nome].data:
        serie.name = {"total_com_diagonal": "Emissões próprias (com diagonal)",
                      "intersetorial_sem_diagonal": "Atribuições a outros setores"}.get(serie.name, serie.name)


### 8.2. Construção e Administração pública: origens e destinos

Detalhamos as atividades **4180 — Construção** e **8400 — Administração pública, defesa e seguridade social**. À esquerda aparecem emissões de outros setores atribuídas à demanda final da atividade; à direita, emissões próprias atribuídas à demanda final de outros setores.

Selecionamos até dez relações positivas de maior peso de cada lado, com desempate por código. As coberturas usam todas as relações intersetoriais como denominador. A diagonal é informada separadamente. Os pesos estão em Gg de CO₂ e incluem requisitos diretos e indiretos: não são apenas compras diretas de insumos. Entradas e saídas são atribuições diferentes; não pressupomos conservação de fluxo pelo setor central.


In [ ]:
# Recortes setoriais calculados diretamente de P, mantendo os pesos originais.
# Coluna: origens emissoras; linha: destinos da demanda final. Diagonal excluída.
estados_destaque = []
for atividade in ["4180", "8400"]:
    entradas_atividade = P[atividade].drop(atividade)
    saidas_atividade = P.loc[atividade].drop(atividade)
    entradas_maiores = entradas_atividade[entradas_atividade > 0].sort_index().sort_values(ascending=False, kind="stable").head(10)
    saidas_maiores = saidas_atividade[saidas_atividade > 0].sort_index().sort_values(ascending=False, kind="stable").head(10)
    # Cobertura do recorte em relação à coluna ou linha completa, sem diagonal.
    total_entrada = entradas_atividade.sum()
    total_saida = saidas_atividade.sum()
    estados_destaque.append({
        "setor": atividade, "descricao": setores[atividade],
        "entradas": [(codigo, setores[codigo], peso) for codigo, peso in entradas_maiores.items()],
        "saidas": [(codigo, setores[codigo], peso) for codigo, peso in saidas_maiores.items()],
        "cobertura_entrada": 100 * entradas_maiores.sum() / total_entrada if total_entrada else 0,
        "cobertura_saida": 100 * saidas_maiores.sum() / total_saida if total_saida else 0,
        "diagonal": P.loc[atividade, atividade],
    })


In [ ]:
# A função externa desenha os recortes já calculados acima.
sankeys_destaque = []
for estado in estados_destaque:
    figura = figura_setor([estado], estado["descricao"])
    sankeys_destaque.append(figura)
    display(figura)

# A página principal referencia este HTML; reexecutar a célula atualiza os dois diagramas.
from plotly.io import to_html
blocos_sankey = [to_html(figura, full_html=False, include_plotlyjs=(i == 0),
    div_id="sankey-" + estados_destaque[i]["setor"], config={"responsive": True, "displaylogo": False})
    for i, figura in enumerate(sankeys_destaque)]
pagina_sankeys = ('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width,initial-scale=1">'
    '<title>Construção e Administração pública — Sankeys</title>'
    '<style>body{margin:0;background:white}</style></head><body>'
    + "".join(blocos_sankey) + '</body></html>')
(dados.RAIZ_PROJETO / "docs" / "sankeys_setores.html").write_text(pagina_sankeys, encoding="utf-8")


## 9. Síntese: volume, diversidade e participação por origem

As frases abaixo são regeneradas a partir dos resultados. Criticidade é discutida como relevância em dimensões distintas; não há escore composto. Um emissor grande pode ter saídas concentradas em poucos destinos, e um emissor menor pode dominar atribuições de determinados destinos. A participação por origem é relativa às atribuições intersetoriais, não às emissões totais do destino.

As hipóteses formuladas no ensaio ainda não foram submetidas a teste estatístico. A concentração entre emissores não substitui a concentração entre arestas, e a diversidade de saídas não mede exposição de um destino a múltiplos emissores. Interpretar essas hipóteses exige definir medidas e critérios próprios antes de formular conclusões. Os níveis absolutos herdam a aproximação monetária e a interpolação descritas na etapa MIP; esta revisão conserva essas hipóteses e os valores calculados.


In [ ]:
conclusoes = []
for universo, linha in concentracao.iterrows():
    conclusoes.append(f"No universo {universo}, os cinco maiores concentram {linha.top5:.1%} e os dez maiores {linha.top10:.1%}; Gini = {linha.gini:.3f}, HHI = {linha.hhi:.3f}.")
maior = metricas["emissoes_totais"].idxmax()
amplo = metricas["destinos_efetivos"].idxmax()
conclusoes.append(f"{maior} — {setores[maior]} lidera as emissões totais ({metricas.loc[maior, 'participacao_total']:.1%}). "
                 f"{amplo} — {setores[amplo]} apresenta o maior diversidade de destinos ({metricas.loc[amplo, 'destinos_efetivos']:.1f} destinos efetivos).")
for papel in ["emissor", "destino"]:
    corr = resumo.iloc[0]["correlacao_rank_" + papel]
    conclusoes.append(f"A correlação dos rankings de PageRank e força para {papel} é {corr:.3f}. "
                     "Quanto mais próxima de 1, menor a mudança de ordenação em relação ao volume.")
mudanca = rankings["promocao_emissor"].abs().idxmax()
conclusoes.append(f"Maior divergência entre força de saída e PageRank emissor: {mudanca} — {setores[mudanca]}, "
                 f"postos {rankings.loc[mudanca, 'forca_saida']:g} e {rankings.loc[mudanca, 'pagerank_emissor']:g}, respectivamente.")
if not principais_emissores.empty:
    relacao = principais_emissores.sort_values("participacao", ascending=False).iloc[0]
    conclusoes.append(f"A maior participação de uma origem em um destino ocorre em {relacao.destino} — {relacao.descricao_destino}: "
                     f"{relacao.participacao:.1%} das atribuições intersetoriais vêm de {relacao.emissor} — {relacao.descricao_emissor} "
                     f"({relacao.peso_gg:,.2f} Gg).")
conclusoes.append("Essas posições identificam relações relevantes para investigação; não demonstram a redução de emissões que resultaria de intervir em um setor.")
for texto in conclusoes: print(texto + "\n")

## 10. Exportação e página interativa

Exporta somente os resultados da rede de produção para `outputs/redes/`: métricas, rankings, concentração, matriz de dependência, resumo, cobertura visual e proveniência. As participações dos CSVs usam frações de 0 a 1. Os recortes de três relações por setor são mostrados no notebook e na página; a matriz completa permite reconstruí-los.

`docs/index.html` referencia `docs/imagens/emissoes_vab.png` como primeiro gráfico e incorpora as demais figuras, tabelas e downloads, sem backend ou recálculo no navegador. A página registra os hashes dos inputs e das fontes das células (sem outputs/contadores), os módulos e versões. O notebook sobrescreve os outputs atuais; publicar por GitHub Pages continua sendo uma etapa separada. Nenhum arquivo da MIP é modificado.

In [ ]:
pasta_outputs = dados.RAIZ_PROJETO / "outputs" / "redes"
pasta_outputs.mkdir(parents=True, exist_ok=True)
tabelas_saida = {"distribuicao_celulas_p": resumo_distribuicao_p, "metricas_producao": metricas, "rankings_producao": rankings,
    "concentracao_producao": concentracao, "dependencia_producao": d,
    "resumo_redes": resumo, "cobertura_visual": cobertura}
for nome, tabela in tabelas_saida.items():
    tabela.to_csv(pasta_outputs / (nome + ".csv"), encoding="utf-8", lineterminator="\n")
# Recortes para consulta; a exportação canônica é a matriz completa de dependência.
tabelas_saida["principais_emissores_por_destino"] = principais_emissores
tabelas_saida["destinos_dependentes_por_emissor"] = destinos_dependentes
registros = [{"tipo": "input", "arquivo_ou_pacote": r["arquivo"], "sha256_ou_versao": r["sha256"]}
             for _, r in entradas.iterrows()]
fonte_notebook = json.loads((dados.RAIZ_PROJETO / "analise_redes_emissoes.ipynb").read_text(encoding="utf-8"))
fontes = json.dumps([c["source"] for c in fonte_notebook["cells"]], ensure_ascii=False).encode("utf-8")
registros.append({"tipo": "fontes das celulas", "arquivo_ou_pacote": "analise_redes_emissoes.ipynb", "sha256_ou_versao": sha256(fontes).hexdigest()})
for arquivo in ["redes/redes.py", "redes/dados.py", "redes/visualizacoes.py", "requirements.txt"]:
    registros.append({"tipo": "codigo", "arquivo_ou_pacote": arquivo,
                      "sha256_ou_versao": sha256((dados.RAIZ_PROJETO / arquivo).read_bytes()).hexdigest()})
for pacote in ["networkx", "pandas", "numpy", "scipy", "plotly"]:
    registros.append({"tipo": "versao", "arquivo_ou_pacote": pacote, "sha256_ou_versao": version(pacote)})
proveniencia = pd.DataFrame(registros)
proveniencia.to_csv(pasta_outputs / "proveniencia.csv", index=False, encoding="utf-8", lineterminator="\n")
exportar_pagina_redes(dados.RAIZ_PROJETO / "docs" / "index.html", figuras, tabelas_saida, conclusoes, proveniencia, cobertura, descricoes_figuras, notas_metodologicas)
print("CSVs: outputs/redes/ | Página interativa: docs/index.html")